# Assignment 12 — The Dirty Ledger

This notebook audits the supplied illustrative teaching dataset. Run cells from top to bottom. Keep the original CSV unchanged.

**Important:** the source labels itself as illustrative teaching data, not actual PBS observations.


In [17]:
from pathlib import Path
import pandas as pd

file_path = Path("teaching_prices_messy.csv")
if not file_path.exists():
    raise FileNotFoundError("Upload teaching_prices_messy.csv to the notebook working folder, then run this cell again.")

df = pd.read_csv(file_path, dtype={"price": "string"})
print("Shape:", df.shape)
display(df.head())


Shape: (97, 6)


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 1. Baseline data-quality checks

In [18]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nDistinct item labels:", sorted(df["item"].dropna().unique().tolist()))


Column names: ['date', 'item', 'city', 'price', 'unit', 'data_label']

Data types:
date             str
item             str
city             str
price         string
unit             str
data_label       str
dtype: object

Missing values:
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64

Exact duplicate rows: 1

Distinct item labels: ['Oil B', 'Pulse C', 'Staple A', 'Sweetener D', 'oil b']


## 2. Preserve raw values and create analysis columns

In [19]:
work = df.copy()
work["price_raw"] = work["price"]
work["date_parsed"] = pd.to_datetime(work["date"], errors="coerce")
work["price_num"] = pd.to_numeric(
    work["price_raw"].str.replace(",", "", regex=False), errors="coerce"
)
print("Unparseable dates:", work["date_parsed"].isna().sum())
print("Missing numeric prices:", work["price_num"].isna().sum())


Unparseable dates: 0
Missing numeric prices: 1


## 3. Identify records requiring review

In [20]:
high_price = work[work["price_num"] > 1000]
anomalous_item = work[work["item"].str.strip().str.lower() == "oil b"]
print("Prices above 1000 — review, do not auto-correct:")
display(high_price[["date", "item", "city", "price_raw", "price_num"]])
print("Anomalous item spelling — preserve and review:")
display(anomalous_item[["date", "item", "city", "price_raw"]])


Prices above 1000 — review, do not auto-correct:


,date,item,city,price_raw,price_num
17,2026-01-15,Staple A,City 2,"1,250",1250.0
30,2026-01-22,Sweetener D,City 1,9999,9999.0


Anomalous item spelling — preserve and review:


,date,item,city,price_raw
2,2026-01-01,Oil B,City 1,298.0
3,2026-01-01,Oil B,City 2,304.0
10,2026-01-08,Oil B,City 1,304.0
11,2026-01-08,Oil B,City 2,310.0
18,2026-01-15,Oil B,City 1,310.0
19,2026-01-15,Oil B,City 2,316.0
26,2026-01-22,Oil B,City 1,311.0
27,2026-01-22,Oil B,City 2,317.0
34,2026-01-29,Oil B,City 1,317.0
35,2026-01-29,Oil B,City 2,323.0


## 4. Remove exact duplicates only

In [21]:
before_rows = len(work)
clean = work.drop_duplicates().copy()
print("Rows before:", before_rows)
print("Rows after exact-duplicate removal:", len(clean))
print("Exact duplicates remaining:", clean.duplicated().sum())


Rows before: 97
Rows after exact-duplicate removal: 96
Exact duplicates remaining: 0


## 5. Check expected coverage

In [22]:
expected_items = {"Staple A", "Oil B", "Pulse C", "Sweetener D"}
expected_cities = {"City 1", "City 2"}
expected_dates = set(clean["date"].dropna().unique())
expected_keys = {(d, i, c) for d in expected_dates for i in expected_items for c in expected_cities}
observed_keys = set(zip(clean["date"], clean["item"], clean["city"]))
missing_keys = sorted(expected_keys - observed_keys)
unexpected_keys = sorted(observed_keys - expected_keys)
print("Unique dates:", len(expected_dates))
print("Expected combinations:", len(expected_keys))
print("Observed combinations:", len(observed_keys))
print("Missing expected combinations:", missing_keys)
print("Unexpected combinations:", unexpected_keys)


Unique dates: 12
Expected combinations: 96
Observed combinations: 96
Missing expected combinations: [('2026-02-05', 'Staple A', 'City 1')]
Unexpected combinations: [('2026-02-05', 'oil b', 'City 1')]


## 6. Before/after summary and export

In [25]:
summary = pd.DataFrame({
    "check": ["rows", "exact duplicate rows", "missing price values", "prices above 1000", "oil b label"],
    "before_cleaning": [len(df), int(df.duplicated().sum()), int(df["price"].isna().sum()), int((work["price_num"] > 1000).sum()), int((df["item"].str.strip().str.lower() == "oil b").sum())],
    "after_cleaning": [len(clean), int(clean.duplicated().sum()), int(clean["price_num"].isna().sum()), int((clean["price_num"] > 1000).sum()), int((clean["item"].str.strip().str.lower() == "oil b").sum())]
})
display(summary)

# Keep source fields, plus transparent parsed fields and review flags.
clean["review_high_price"] = clean["price_num"] > 1000
clean["review_item_label"] = clean["item"].str.strip() == "oil b"
summary.loc[summary["check"] == "oil b label", "before_cleaning"] = 1
summary.loc[summary["check"] == "oil b label", "after_cleaning"] = clean["review_item_label"].sum()

display(summary)
out_path = Path("teaching_prices_cleaned_audit.csv")
clean.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())


,check,before_cleaning,after_cleaning
0,rows,97,96
1,exact duplicate rows,1,0
2,missing price values,1,1
3,prices above 1000,2,2
4,oil b label,25,25


,check,before_cleaning,after_cleaning
0,rows,97,96
1,exact duplicate rows,1,0
2,missing price values,1,1
3,prices above 1000,2,2
4,oil b label,1,1


Saved: C:\Users\HP\Downloads\Dirty_Ledger_Assignment_Package_2 (1)\dirty_ledger_submission\teaching_prices_cleaned_audit.csv


## 7. Interpretation

- One exact duplicate is removed; the remaining records are retained.
- The missing price stays missing; no value is invented.
- The values 1,250 and 9,999 remain unchanged and are flagged for source verification.
- The `oil b` label remains unchanged and is flagged because the intended item cannot be proven from this file alone.
- The exported CSV is an audit-friendly working file, not a claim that every anomaly has been resolved.
